In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.preprocessing import RobustScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score, precision_score, recall_score, f1_score, make_scorer
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

def regularization_analysis_model2():
    """
    Regularization analysis for Model 2 (Práticas de Alimentação)
    Inclui correção de tipos de dados e nested cross-validation
    """
    
    # Load training data do Model 2
    train_path = '/Users/marcelosilva/Desktop/early-obesity-prediction/D-Train-Test Split/MODEL2TRAIN.csv'
    df = pd.read_csv(train_path)
    
    print("="*80)
    print("COMPARATIVE REGULARIZATION ANALYSIS - MODELO 2")
    print("="*80)
    
    print(f"Original dataset:")
    print(f"  - Dimensions: {df.shape}")
    print(f"  - Colunas: {list(df.columns)}")
    
    # Identificar variables que devem ser binárias mas estão como float
    binary_vars = [
        'doou_leite_banco', 'recebeu_leite_banco', 'amamentou_outra_crianca',
        'usou_mamadeira', 'deixou_amamentar_por_outra', 'busca_info_aleitamento',
        'deu_outros_liquidos', 'k15_recebeu', 'k11_amamentou', 'k03_prenatal',
        'usou_utensilios_amamentacao'
    ]
    
    # Fix data types
    print(f"\n🔧 CORREÇÃO DE TIPOS DE DADOS:")
    print(f"Variáveis binárias identificadas: {len(binary_vars)}")
    
    for var in binary_vars:
        if var in df.columns:
            unique_vals = df[var].dropna().unique()
            print(f"  - {var}: valores únicos = {sorted(unique_vals)}")
            
            # Converter para binário (0 e 1)
            df[var] = df[var].astype(int)
    
    # Verificar dados após correção
    print(f"\nTipos de dados após correção:")
    print(df.dtypes)
    
    # Prepare data
    target_cols = ['desnutrido', 'eutrofico', 'sobrepeso', 'obeso']
    y = df['obeso'].copy()
    
    # Features: excluir id_anon e variables target
    feature_cols = [col for col in df.columns if col not in target_cols + ['id_anon']]
    X = df[feature_cols].copy()
    
    print(f"\nDataset de treinamento:")
    print(f"  - Observations: {len(df):,}")
    print(f"  - Features: {len(feature_cols)}")
    print(f"  - Obese: {y.sum()} ({y.mean()*100:.1f}%)")
    print(f"  - Não-obesos: {(1-y).sum()} ({(1-y.mean())*100:.1f}%)")
    
    # Categorizar tipos de variables
    variable_groups = {
        'peso_materno': ['k08_quilos', 'k07_peso_final', 'k06_peso_engravidar'],
        'prenatal': ['k11_amamentou', 'k03_prenatal', 'k04_prenatal_semanas'],
        'amamentacao_inicial': ['tempo_primeira_mamada_horas', 'usou_utensilios_amamentacao', 'dias_aleitamento_exclusivo'],
        'banco_leite': ['doou_leite_banco', 'recebeu_leite_banco'],
        'praticas_alimentares': ['amamentou_outra_crianca', 'usou_mamadeira', 'deixou_amamentar_por_outra'],
        'comportamento': ['busca_info_aleitamento', 'deu_outros_liquidos', 'k15_recebeu']
    }
    
    print(f"\nGrupos de variables identificados:")
    for grupo, features in variable_groups.items():
        available_features = [f for f in features if f in X.columns]
        print(f"  - {grupo.replace('_', ' ').title()}: {len(available_features)} variables")
    
    # Cross-validation settings
    outer_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    inner_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    
    # Parameters for grid search
    param_grids = {
        'lasso': {
            'model__C': [0.1, 1.0, 10.0, 100.0, 1000.0],
            'model__penalty': ['l1']
        },
        'ridge': {
            'model__C': [0.1, 1.0, 10.0, 100.0, 1000.0],
            'model__penalty': ['l2']
        },
        'elastic_net': {
            'model__C': [0.1, 1.0, 10.0, 100.0, 1000.0],
            'model__penalty': ['elasticnet'],
            'model__l1_ratio': [0.1, 0.3, 0.5, 0.7, 0.9]
        }
    }
    
    def calculate_confidence_intervals(scores, confidence=0.95):
        """Calculates confidence intervals using t-distribution"""
        n = len(scores)
        mean_score = np.mean(scores)
        std_err = stats.sem(scores)
        h = std_err * stats.t.ppf((1 + confidence) / 2., n-1)
        return mean_score, mean_score - h, mean_score + h
    
    def evaluate_regularization_method(X_data, y_data, model, param_grid, method_name):
        """Evaluates a regularization method with nested cross-validation"""
        
        print(f"\n[SEARCH] Evaluating {method_name}...")
        
        # Pipeline with preprocessing
        pipeline = Pipeline([
            ('scaler', RobustScaler()),
            ('model', model)
        ])
        
        # Nested cross-validation
        results = []
        best_params_list = []
        selected_features_list = []
        
        for fold, (train_idx, val_idx) in enumerate(outer_cv.split(X_data, y_data)):
            X_train_fold, X_val_fold = X_data.iloc[train_idx], X_data.iloc[val_idx]
            y_train_fold, y_val_fold = y_data.iloc[train_idx], y_data.iloc[val_idx]
            
            # Grid search no inner CV
            grid_search = GridSearchCV(
                pipeline, param_grid, cv=inner_cv, 
                scoring='roc_auc', n_jobs=-1, verbose=0
            )
            
            grid_search.fit(X_train_fold, y_train_fold)
            best_params_list.append(grid_search.best_params_)
            
            # Avaliar no validation fold
            y_pred_proba = grid_search.predict_proba(X_val_fold)[:, 1]
            y_pred = grid_search.predict(X_val_fold)
            
            # Calcular métricas
            fold_results = {
                'auc': roc_auc_score(y_val_fold, y_pred_proba),
                'precision': precision_score(y_val_fold, y_pred, zero_division=0),
                'recall': recall_score(y_val_fold, y_pred, zero_division=0),
                'f1': f1_score(y_val_fold, y_pred, zero_division=0)
            }
            results.append(fold_results)
            
            # Features selecionadas (para Lasso e Elastic Net)
            if hasattr(grid_search.best_estimator_.named_steps['model'], 'coef_'):
                coefs = grid_search.best_estimator_.named_steps['model'].coef_
                if coefs.ndim > 1:
                    coefs = coefs.flatten()
                selected_features = X_data.columns[coefs != 0].tolist()
                selected_features_list.append(selected_features)
        
        # Consolidate results
        metrics_summary = {}
        for metric in ['auc', 'precision', 'recall', 'f1']:
            scores = [r[metric] for r in results]
            mean_score, ci_lower, ci_upper = calculate_confidence_intervals(scores)
            metrics_summary[metric] = {
                'mean': mean_score,
                'std': np.std(scores),
                'ci_lower': ci_lower,
                'ci_upper': ci_upper,
                'scores': scores
            }
        
        return {
            'method': method_name,
            'metrics': metrics_summary,
            'best_params': best_params_list,
            'selected_features': selected_features_list,
            'n_features_selected': [len(sf) for sf in selected_features_list] if selected_features_list else None
        }
    
    # =======================================================================
    # AVALIAÇÃO DOS MÉTODOS DE REGULARIZAÇÃO
    # =======================================================================
    print(f"\n" + "="*60)
    print("AVALIAÇÃO DE MÉTODOS DE REGULARIZAÇÃO")
    print("="*60)
    
    results = {}
    
    # Lasso
    lasso_results = evaluate_regularization_method(
        X, y, LogisticRegression(random_state=42, max_iter=10000, penalty='l1', solver='liblinear', class_weight='balanced'), 
        param_grids['lasso'], 'Lasso'
    )
    results['lasso'] = lasso_results
    
    # Ridge  
    ridge_results = evaluate_regularization_method(
        X, y, LogisticRegression(random_state=42, max_iter=10000, penalty='l2', class_weight='balanced'), 
        param_grids['ridge'], 'Ridge'
    )
    results['ridge'] = ridge_results
    
    # Elastic Net
    elastic_results = evaluate_regularization_method(
        X, y, LogisticRegression(random_state=42, max_iter=10000, penalty='elasticnet', solver='saga', class_weight='balanced'), 
        param_grids['elastic_net'], 'Elastic Net'
    )
    results['elastic_net'] = elastic_results
    
    # =======================================================================
    # COMPARATIVE RESULTS
    # =======================================================================
    print(f"\n" + "="*80)
    print("COMPARATIVE RESULTS - MODELO 2")
    print("="*80)
    
    print(f"{'Método':<15} {'AUC-ROC':<25} {'Precision':<25} {'Recall':<25} {'F1-Score':<25}")
    print("-" * 115)
    
    for method_key, result in results.items():
        method_name = result['method']
        auc = result['metrics']['auc']
        precision = result['metrics']['precision']
        recall = result['metrics']['recall']
        f1 = result['metrics']['f1']
        
        auc_str = f"{auc['mean']:.3f} ({auc['ci_lower']:.3f}-{auc['ci_upper']:.3f})"
        prec_str = f"{precision['mean']:.3f} ({precision['ci_lower']:.3f}-{precision['ci_upper']:.3f})"
        rec_str = f"{recall['mean']:.3f} ({recall['ci_lower']:.3f}-{recall['ci_upper']:.3f})"
        f1_str = f"{f1['mean']:.3f} ({f1['ci_lower']:.3f}-{f1['ci_upper']:.3f})"
        
        print(f"{method_name:<15} {auc_str:<25} {prec_str:<25} {rec_str:<25} {f1_str:<25}")
    
    # Analysis de feature selection
    print(f"\n" + "="*60)
    print("FEATURE SELECTION ANALYSIS")
    print("="*60)
    
    for method_key, result in results.items():
        if result['n_features_selected']:
            n_features = result['n_features_selected']
            method_name = result['method']
            print(f"  {method_name}: {np.mean(n_features):.1f} ± {np.std(n_features):.1f} features selected")
    
    # Identify best method
    print(f"\n" + "="*60)
    print("METHODOLOGICAL RECOMMENDATION")
    print("="*60)
    
    all_results = []
    for method_key, result in results.items():
        all_results.append({
            'method': result['method'],
            'auc_mean': result['metrics']['auc']['mean'],
            'auc_ci_width': result['metrics']['auc']['ci_upper'] - result['metrics']['auc']['ci_lower'],
            'auc_std': result['metrics']['auc']['std']
        })
    
    # Sort by mean AUC
    all_results.sort(key=lambda x: x['auc_mean'], reverse=True)
    
    print(f"Ranking by mean AUC-ROC:")
    for i, result in enumerate(all_results, 1):
        print(f"{i}. {result['method']}: AUC = {result['auc_mean']:.3f} ± {result['auc_std']:.3f}")
    
    best_method = all_results[0]
    print(f"\nRECOMMENDED METHOD: {best_method['method']}")
    print(f"  - AUC-ROC: {best_method['auc_mean']:.3f}")
    print(f"  - Estabilidade (std): {best_method['auc_std']:.3f}")
    
    # Comparison with Model 1
    print(f"\n" + "="*60)
    print("COMPARAÇÃO COM MODELO 1")
    print("="*60)
    print(f"Model 1 (Fatores Demográficos/Perinatais):")
    print(f"  - Melhor AUC-ROC: 0.543 (Lasso)")
    print(f"  - Features: 27 variables")
    print(f"  - Amostra: 6,588 observations")
    
    print(f"\nModel 2 (Práticas de Alimentação):")
    print(f"  - Melhor AUC-ROC: {best_method['auc_mean']:.3f} ({best_method['method']})")
    print(f"  - Features: {len(feature_cols)} variables")
    print(f"  - Amostra: {len(df):,} observations")
    
    if best_method['auc_mean'] > 0.543:
        print(f"\n[OK] Model 2 demonstra melhor capacidade preditiva")
    else:
        print(f"\n⚠ Ambos os modelos apresentam limitações preditivas similares")
    
    return results, best_method

# Execute analysis
if __name__ == "__main__":
    model2_results, best_model2 = regularization_analysis_model2()

COMPARATIVE REGULARIZATION ANALYSIS - MODELO 2
Original dataset:
  - Dimensions: (4510, 22)
  - Colunas: ['id_anon', 'k08_quilos', 'k07_peso_final', 'k06_peso_engravidar', 'k11_amamentou', 'k03_prenatal', 'k04_prenatal_semanas', 'tempo_primeira_mamada_horas', 'usou_utensilios_amamentacao', 'dias_aleitamento_exclusivo', 'doou_leite_banco', 'recebeu_leite_banco', 'amamentou_outra_crianca', 'usou_mamadeira', 'deixou_amamentar_por_outra', 'busca_info_aleitamento', 'deu_outros_liquidos', 'k15_recebeu', 'desnutrido', 'eutrofico', 'sobrepeso', 'obeso']

🔧 CORREÇÃO DE TIPOS DE DADOS:
Variáveis binárias identificadas: 11
  - doou_leite_banco: valores únicos = [0.0, 1.0]
  - recebeu_leite_banco: valores únicos = [0.0, 1.0]
  - amamentou_outra_crianca: valores únicos = [0.0, 1.0]
  - usou_mamadeira: valores únicos = [0.0, 1.0]
  - deixou_amamentar_por_outra: valores únicos = [0.0, 1.0]
  - busca_info_aleitamento: valores únicos = [0.0, 1.0]
  - deu_outros_liquidos: valores únicos = [0.0, 1.0]
  -